# Bearing Fault Classification — Clean Core Workflow

This notebook retains the substantive workflow for the selected **Extra Trees v2** model: prepare the window feature table, validate the fault-instance grouping, compare candidate models, compare median versus median + window-IQR features, train/save the selected model, and define inference for a new `.mat` recording.

It intentionally excludes separate v1 branches, repeated debugging and artifact audits, exploratory spectrum plots, and external-domain v3/specialist experiments. Those are not needed to reproduce the selected v2 training path.

**Run cells from top to bottom.** The full grouped validation and feature-representation comparison are included; they can take longer than simply loading the saved model.

**Reproduction caveat:** the grouped validation is rerun and its resulting metrics are written to the reproduced artifact. Performance is not certified identical until this notebook is executed against the project data and its output is compared with the historical run. The original saved v2 model is not overwritten.


## Data and validation caveat

The v2 development table includes `cwru_additional_development_features.csv`, which is built from `independent_holdout_cwru/mat`. Therefore, evaluating the final v2 artifact on those same recordings is **not an untouched independent test**. The grouped cross-validation below is a development estimate; report truly external performance separately.

In [10]:
%pip install -q numpy pandas scipy scikit-learn joblib


## 1. Connect Drive and prepare the feature table

In [11]:

# ==============================================================
# CORE SETUP: load the project, extractor, and the three window-feature tables.
# ==============================================================

from pathlib import Path
import re
import importlib.util
import numpy as np
import pandas as pd
import joblib

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    confusion_matrix
)
from IPython.display import display

# --------------------------------------------------------------
# 1. Restore project paths and feature extractor
# --------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/BTP_Bearing_Fault_Project")

if not ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

if not ROOT.exists():
    raise FileNotFoundError(f"Google Drive project not found: {ROOT}")

train_path = ROOT / "results_multiband/tables/train_multiband_features.csv"
severity_path = ROOT / "results_multiband/tables/severity_multiband_features.csv"
test_dir = ROOT / "independent_holdout_cwru" / "mat"

for p in [train_path, severity_path, test_dir]:
    if not p.exists():
        raise FileNotFoundError(f"Required project path is missing: {p}")

spec = importlib.util.spec_from_file_location(
    "bearing_extractor_grouped_validation",
    ROOT / "src" / "extract_multiband_envelope_features.py"
)
extractor = importlib.util.module_from_spec(spec)
spec.loader.exec_module(extractor)

# The extractor supports only the original defect-size codes.
# Extend this parser to include 0.028-inch CWRU files.
def infer_diameter_extended(stem):
    match = re.match(
        r"^(?:IR|B|OR)(007|014|021|028)",
        str(stem),
        re.IGNORECASE
    )
    return int(match.group(1)) / 1000 if match else np.nan

extractor.infer_diameter = infer_diameter_extended

# Handle missing RPM only for the downloaded CWRU holdout directory.
# These are approximate nominal speeds, not measured values.
base_reader = extractor.read_signals_and_rpm

def read_signals_with_holdout_rpm(path):
    path = Path(path)

    try:
        return base_reader(path)
    except ValueError as exc:
        if "RPM value not found" not in str(exc):
            raise

        if not path.resolve().is_relative_to(test_dir.resolve()):
            raise

        load_hp = extractor.infer_load(path.stem)
        nominal_rpm = {
            0: 1797.0, 1: 1772.0, 2: 1750.0, 3: 1730.0
        }

        if not np.isfinite(load_hp) or int(load_hp) not in nominal_rpm:
            raise ValueError(
                f"Cannot infer RPM for {path.name}"
            ) from exc

        mat = extractor.loadmat(str(path))
        keys = sorted(k for k in mat if k.endswith("_DE_time"))

        if not keys:
            raise ValueError(f"No DE signal in {path.name}")

        signals = []
        for key in keys:
            signal = np.asarray(mat[key], dtype=float).ravel()
            if not np.isfinite(signal).all():
                raise ValueError(f"Non-finite signal in {path.name}")
            signals.append((key, signal))

        rpm = nominal_rpm[int(load_hp)]
        print(f"Approximate RPM used: {path.name} -> {rpm:.0f}")
        return signals, rpm

extractor.read_signals_and_rpm = read_signals_with_holdout_rpm

# --------------------------------------------------------------
# 2. Load existing tables; extract added CWRU features only if absent
# --------------------------------------------------------------

train_windows = pd.read_csv(train_path)
severity_windows = pd.read_csv(severity_path)

feature_cols = [
    c for c in train_windows.columns
    if c.startswith(("bp_", "env_")) and c in severity_windows.columns
]

if len(feature_cols) != 76:
    raise ValueError(
        f"Expected 76 common signal features, found {len(feature_cols)}"
    )

extra_path = (
    ROOT / "results_multiband/tables/"
    "cwru_additional_development_features.csv"
)

# Re-extract only when needed. The existing source feature tables
# are left unchanged.
if extra_path.exists():
    extra_windows = pd.read_csv(extra_path)
    if not set(feature_cols).issubset(extra_windows.columns):
        extra_windows = None
else:
    extra_windows = None

if extra_windows is None:
    extra_windows = extractor.process_directory(
        test_dir,
        extra_path,
        "cwru_additional_development",
        extractor.make_filter_bank()
    )

# Restore extended defect-size metadata if the CSV was created before
# the 0.028 parser patch.
extra_windows["defect_diameter_in"] = (
    extra_windows["filename"].astype(str)
    .str.extract(r"^(?:IR|B|OR)(007|014|021|028)", expand=False)
    .pipe(pd.to_numeric, errors="coerce") / 1000
)

classes = ["ball_fault", "inner_race_fault", "outer_race_fault"]

combined = pd.concat(
    [train_windows, severity_windows, extra_windows],
    ignore_index=True,
    sort=False
)

combined["label"] = combined["label"].astype(str).str.strip()
combined["defect_diameter_in"] = pd.to_numeric(
    combined["defect_diameter_in"], errors="coerce"
).round(3)

combined = combined[combined["label"].isin(classes)].copy()

for col in feature_cols:
    combined[col] = pd.to_numeric(combined[col], errors="coerce")

combined[feature_cols] = combined[feature_cols].replace(
    [np.inf, -np.inf], np.nan
)

# Group recordings from the same fault instance across loads.
# Examples: B007_0.mat -> B007; OR007_3_0.mat -> OR007_3.
combined["fault_instance_group"] = (
    combined["filename"].astype(str)
    .str.replace(r"_[0-3]\.mat$", "", regex=True)
)

record_meta = combined[
    ["filename", "label", "fault_instance_group", "defect_diameter_in"]
].drop_duplicates()

if record_meta["filename"].duplicated().any():
    raise ValueError("Inconsistent recording metadata detected.")

if record_meta.groupby("fault_instance_group")["label"].nunique().gt(1).any():
    raise ValueError("Conflicting labels within a fault-instance group.")

print("\nRecordings:", record_meta["filename"].nunique())
print("Fault-instance groups:", record_meta["fault_instance_group"].nunique())
print("\nRecordings by class and diameter:")
display(
    record_meta.groupby(["defect_diameter_in", "label"])
    .size().unstack(fill_value=0)
)

# --------------------------------------------------------------



Recordings: 60
Fault-instance groups: 18

Recordings by class and diameter:


label,ball_fault,inner_race_fault,outer_race_fault
defect_diameter_in,,,
0.007,4,4,12
0.014,4,4,4
0.021,4,4,12
0.028,4,4,0


## 2. Verify independent recording and fault-instance groups

**Grouping note:** The recording-group count displayed in Section 1 is preliminary. This section applies canonical filename normalization and verifies the final dataset composition: 60 recordings across 15 fault-instance groups. These canonical groups are used for grouped cross-validation.

In [12]:
# Normalize recording names and group all loads from the same physical fault instance.
import re


def canonical_recording_name(value):
    name = str(value).strip().replace("\\", "/").rsplit("/", 1)[-1]
    return re.sub(r"(?i)\.mat$", "", name).strip().upper()


def canonical_fault_group(filename, label):
    stem = canonical_recording_name(filename)
    stem = re.sub(r"_[0-3]$", "", stem)
    return f"{str(label).strip().lower()}::{stem}"


combined = combined.copy()
combined["filename_clean"] = combined["filename"].map(canonical_recording_name)
combined["fault_instance_group"] = [
    canonical_fault_group(filename, label)
    for filename, label in zip(combined["filename"], combined["label"])
]
combined["label"] = combined["label"].astype(str).str.strip()
combined["defect_diameter_in"] = pd.to_numeric(
    combined["defect_diameter_in"], errors="coerce"
).round(3)

metadata_check = combined.groupby("filename_clean").agg(
    label_count=("label", "nunique"),
    size_count=("defect_diameter_in", "nunique"),
    group_count=("fault_instance_group", "nunique"),
)
if ((metadata_check > 1).any().any()):
    raise ValueError("Conflicting recording metadata after filename normalization.")

record_meta = combined.groupby("filename_clean", as_index=False).agg(
    label=("label", "first"),
    defect_diameter_in=("defect_diameter_in", "first"),
    fault_instance_group=("fault_instance_group", "first"),
)
print("Unique recordings:", len(record_meta))
print("Unique fault-instance groups:", record_meta["fault_instance_group"].nunique())

# The saved v2 experiment used 60 recordings from 15 fault-instance groups.
if len(record_meta) != 60 or record_meta["fault_instance_group"].nunique() != 15:
    raise ValueError(
        "Unexpected data composition. Inspect record_meta and group assignments "
        "before treating any CV score as valid."
    )


Unique recordings: 60
Unique fault-instance groups: 15


## 3. Define model candidates

In [13]:
import sys
from pathlib import Path

sys.path.insert(0, str(ROOT / "src"))
components_path = ROOT / "src" / "model_components.py"
if not components_path.is_file():
    raise FileNotFoundError(f"Required model component is missing: {components_path}")
import model_components
CorrelationFilter = model_components.CorrelationFilter
print("Using persistent CorrelationFilter implementation:", components_path)


Using persistent CorrelationFilter implementation: /content/drive/MyDrive/BTP_Bearing_Fault_Project/src/model_components.py


In [14]:

# Controlled algorithm comparison using corrected fault-instance groups.
# Reuses: combined, feature_cols, CorrelationFilter, ROOT.
# All windows from a fault-instance group stay in the same fold.

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

X_data = combined[feature_cols].reset_index(drop=True)
y_data = combined["label"].astype(str).reset_index(drop=True)
meta = combined[
    ["filename", "label", "fault_instance_group"]
].reset_index(drop=True)
groups = combined["fault_instance_group"].reset_index(drop=True)

def build_candidate(estimator, scale=False):
    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("variance", VarianceThreshold()),
        ("correlation", CorrelationFilter(threshold=0.98)),
        ("select", SelectKBest(score_func=f_classif, k=15))
    ]
    if scale:
        steps.append(("scaler", RobustScaler()))
    steps.append(("model", estimator))
    return Pipeline(steps)

candidates = {
    "Random Forest": build_candidate(
        RandomForestClassifier(
            n_estimators=300, max_depth=3, min_samples_leaf=2,
            max_features="sqrt", class_weight="balanced",
            random_state=42, n_jobs=-1
        )
    ),
    "Extra Trees": build_candidate(
        ExtraTreesClassifier(
            n_estimators=300, max_depth=3, min_samples_leaf=2,
            max_features="sqrt", class_weight="balanced",
            random_state=42, n_jobs=-1
        )
    ),
    "Linear SVM": build_candidate(
        LinearSVC(C=0.1, class_weight="balanced", max_iter=20000),
        scale=True
    ),
    "RBF SVM": build_candidate(
        SVC(C=1.0, kernel="rbf", class_weight="balanced"),
        scale=True
    ),
    "Logistic Regression": build_candidate(
        LogisticRegression(
            C=0.1, class_weight="balanced", max_iter=5000
        ),
        scale=True
    )
}



## 4. Compare models with recording-level grouped cross-validation

In [15]:

# RECORDING-LEVEL TRAINING WITH STRICT FAULT-GROUP VALIDATION
# Reuses existing: combined, feature_cols, candidates, ROOT, clone
# One row per recording; all loads from one fault instance stay together.

# 1. Aggregate window features to one median row per recording.
recording_data = (
    combined.groupby("filename_clean", as_index=False)
    .agg({
        "label": "first",
        "fault_instance_group": "first",
        "defect_diameter_in": "first",
        **{c: "median" for c in feature_cols}
    })
)

print("Recording-level rows:", len(recording_data))
print("Independent fault-instance groups:",
      recording_data["fault_instance_group"].nunique())

if len(recording_data) != 60:
    raise ValueError(
        f"Expected 60 recordings; found {len(recording_data)}."
    )

if recording_data["fault_instance_group"].nunique() != 15:
    raise ValueError("Expected 15 fault-instance groups.")

# 2. Leave one entire fault-instance group out per fold.
X_rec = recording_data[feature_cols].replace(
    [np.inf, -np.inf], np.nan
)
y_rec = recording_data["label"].astype(str)
groups_rec = recording_data["fault_instance_group"].astype(str)

logo = LeaveOneGroupOut()
oof_recording = []
fold_metrics_recording = []

for model_name, prototype in candidates.items():
    for fold, (train_idx, test_idx) in enumerate(
        logo.split(X_rec, y_rec, groups=groups_rec), start=1
    ):
        train_groups = set(groups_rec.iloc[train_idx])
        test_groups = set(groups_rec.iloc[test_idx])

        if train_groups & test_groups:
            raise ValueError("Fault-instance leakage detected.")

        fitted = clone(prototype)
        fitted.fit(X_rec.iloc[train_idx], y_rec.iloc[train_idx])
        pred = fitted.predict(X_rec.iloc[test_idx])

        for idx, predicted in zip(test_idx, pred):
            actual = y_rec.iloc[idx]

            oof_recording.append({
                "model": model_name,
                "filename": recording_data.iloc[idx]["filename_clean"],
                "fault_instance_group": groups_rec.iloc[idx],
                "defect_diameter_in":
                    recording_data.iloc[idx]["defect_diameter_in"],
                "actual": actual,
                "predicted": predicted,
                "correct": actual == predicted
            })

        test_actual = y_rec.iloc[test_idx]
        fold_metrics_recording.append({
            "model": model_name,
            "fold": fold,
            "accuracy": accuracy_score(test_actual, pred),
            "balanced_accuracy": balanced_accuracy_score(
                test_actual, pred
            ),
            "macro_f1": f1_score(
                test_actual, pred,
                labels=classes,
                average="macro",
                zero_division=0
            )
        })

oof_rec = pd.DataFrame(oof_recording)
fold_rec = pd.DataFrame(fold_metrics_recording)

# 3. Pooled out-of-fold metrics, one prediction per recording.
summary_rec = (
    oof_rec.groupby("model")
    .apply(
        lambda g: pd.Series({
            "accuracy": accuracy_score(g["actual"], g["predicted"]),
            "balanced_accuracy": balanced_accuracy_score(
                g["actual"], g["predicted"]
            ),
            "macro_f1": f1_score(
                g["actual"], g["predicted"],
                labels=classes,
                average="macro",
                zero_division=0
            ),
            "correct_recordings": int(g["correct"].sum()),
            "total_recordings": g["filename"].nunique()
        }),
        include_groups=False
    )
    .reset_index()
    .sort_values(
        ["balanced_accuracy", "macro_f1"],
        ascending=False
    )
)

print("\n===== RECORDING-LEVEL GROUPED RESULTS =====")
display(summary_rec.round(3))

print("\n===== PERFORMANCE BY DEFECT SIZE =====")
size_rows = []

for (model_name, size), part in oof_rec.groupby(
    ["model", "defect_diameter_in"], dropna=False
):
    size_rows.append({
        "model": model_name,
        "defect_diameter_in": size,
        "accuracy": accuracy_score(part["actual"], part["predicted"]),
        "recordings": len(part)
    })

display(
    pd.DataFrame(size_rows)
    .sort_values(["model", "defect_diameter_in"])
    .round(3)
)

# 4. Confusion matrix for the top recording-level candidate.
best_name = summary_rec.iloc[0]["model"]
best_pred = oof_rec[oof_rec["model"] == best_name]

print("\nTop candidate:", best_name)
display(pd.DataFrame(
    confusion_matrix(
        best_pred["actual"], best_pred["predicted"], labels=classes
    ),
    index=[f"Actual: {c}" for c in classes],
    columns=[f"Predicted: {c}" for c in classes]
))

# 5. Save diagnostic outputs.
OUT = ROOT / "colab_model_diagnostics" / "recording_level_grouped_v1"
OUT.mkdir(parents=True, exist_ok=True)

summary_rec.to_csv(OUT / "model_summary.csv", index=False)
oof_rec.to_csv(OUT / "oof_predictions.csv", index=False)
fold_rec.to_csv(OUT / "fold_metrics.csv", index=False)

print("\nSaved results to:", OUT)


Recording-level rows: 60
Independent fault-instance groups: 15


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarnin


===== RECORDING-LEVEL GROUPED RESULTS =====


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


,model,accuracy,balanced_accuracy,macro_f1,correct_recordings,total_recordings
0,Extra Trees,0.783,0.765,0.754,47.0,60.0
4,Random Forest,0.767,0.744,0.743,46.0,60.0
1,Linear SVM,0.767,0.744,0.739,46.0,60.0
3,RBF SVM,0.750,0.723,0.726,45.0,60.0
2,Logistic Regression,0.733,0.702,0.699,44.0,60.0



===== PERFORMANCE BY DEFECT SIZE =====


,model,defect_diameter_in,accuracy,recordings
0,Extra Trees,0.007,1.000,20
1,Extra Trees,0.014,0.333,12
2,Extra Trees,0.021,0.950,20
3,Extra Trees,0.028,0.500,8
4,Linear SVM,0.007,1.000,20
5,Linear SVM,0.014,0.417,12
6,Linear SVM,0.021,0.850,20
7,Linear SVM,0.028,0.500,8
8,Logistic Regression,0.007,0.950,20
9,Logistic Regression,0.014,0.333,12



Top candidate: Extra Trees


,Predicted: ball_fault,Predicted: inner_race_fault,Predicted: outer_race_fault
Actual: ball_fault,15,1,0
Actual: inner_race_fault,5,8,3
Actual: outer_race_fault,4,0,24



Saved results to: /content/drive/MyDrive/BTP_Bearing_Fault_Project/colab_model_diagnostics/recording_level_grouped_v1


## 5. Compare median-only versus median + window-IQR features

In [16]:

# TARGETED EXPERIMENT: median-only versus median + window IQR
# Reuses the current notebook variables and the existing Extra Trees pipeline.

from sklearn.base import clone
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

if "combined" not in globals() or "feature_cols" not in globals():
    raise RuntimeError("Run the corrected grouped-validation setup first.")

if "Extra Trees" not in candidates:
    raise RuntimeError("The existing Extra Trees pipeline was not found.")

# One metadata row per independent recording
rec_meta = (
    combined.groupby("filename_clean", as_index=False)
    .agg(
        label=("label", "first"),
        group=("fault_instance_group", "first"),
        size=("defect_diameter_in", "first")
    )
    .sort_values("filename_clean")
    .reset_index(drop=True)
)

# Recording median features
medians = (
    combined.groupby("filename_clean")[feature_cols]
    .median()
    .reindex(rec_meta["filename_clean"])
    .reset_index(drop=True)
)

# Window-to-window variability per recording
q25 = combined.groupby("filename_clean")[feature_cols].quantile(0.25)
q75 = combined.groupby("filename_clean")[feature_cols].quantile(0.75)
iqr = (q75 - q25).reindex(rec_meta["filename_clean"]).reset_index(drop=True)
iqr.columns = [f"{c}__window_iqr" for c in feature_cols]

X_median = medians.replace([np.inf, -np.inf], np.nan)
X_median_iqr = pd.concat([X_median, iqr], axis=1).replace(
    [np.inf, -np.inf], np.nan
)

y_rec = rec_meta["label"].reset_index(drop=True)
groups_rec = rec_meta["group"].reset_index(drop=True)
sizes_rec = rec_meta["size"].reset_index(drop=True)

assert len(rec_meta) == 60
assert groups_rec.nunique() == 15
assert rec_meta["filename_clean"].is_unique

print("Independent recordings:", len(rec_meta))
print("Fault-instance groups:", groups_rec.nunique())
print("Median features:", X_median.shape[1])
print("Median + IQR features:", X_median_iqr.shape[1])

# Use identical leave-one-fault-instance-group-out splits.
logo = LeaveOneGroupOut()
splits = list(logo.split(X_median, y_rec, groups=groups_rec))

results = {}
predictions = {}

feature_variants = {
    "median_only": X_median,
    "median_plus_IQR": X_median_iqr
}

for variant_name, X_variant in feature_variants.items():
    oof_pred = np.empty(len(rec_meta), dtype=object)

    for train_idx, test_idx in splits:
        assert not (
            set(groups_rec.iloc[train_idx])
            & set(groups_rec.iloc[test_idx])
        )

        model = clone(candidates["Extra Trees"])
        model.fit(X_variant.iloc[train_idx], y_rec.iloc[train_idx])
        oof_pred[test_idx] = model.predict(X_variant.iloc[test_idx])

    predictions[variant_name] = oof_pred

    results[variant_name] = {
        "accuracy": accuracy_score(y_rec, oof_pred),
        "balanced_accuracy": balanced_accuracy_score(y_rec, oof_pred),
        "macro_f1": f1_score(
            y_rec, oof_pred,
            labels=sorted(y_rec.unique()),
            average="macro",
            zero_division=0
        ),
        "correct": int(np.sum(oof_pred == y_rec.to_numpy())),
        "total": len(y_rec)
    }

comparison_iqr = (
    pd.DataFrame(results).T
    .rename_axis("feature_representation")
    .reset_index()
    .sort_values("balanced_accuracy", ascending=False)
)

print("\n========== FEATURE REPRESENTATION COMPARISON ==========")
display(comparison_iqr.round(3))

# Compare performance by defect size for both representations.
size_rows = []

for variant_name, pred in predictions.items():
    for size in sorted(sizes_rec.dropna().unique()):
        mask = np.isclose(sizes_rec.astype(float), float(size))
        actual = y_rec.to_numpy()[mask]
        predicted = pred[mask]

        size_rows.append({
            "feature_representation": variant_name,
            "defect_size": size,
            "recordings": int(mask.sum()),
            "accuracy": accuracy_score(actual, predicted)
        })

size_comparison = pd.DataFrame(size_rows)

print("\n========== PERFORMANCE BY DEFECT SIZE ==========")
display(size_comparison.round(3))

OUT = ROOT / "colab_model_diagnostics" / "median_vs_iqr_v1"
OUT.mkdir(parents=True, exist_ok=True)

comparison_iqr.to_csv(OUT / "overall_comparison.csv", index=False)
size_comparison.to_csv(OUT / "size_comparison.csv", index=False)

for name, pred in predictions.items():
    saved = rec_meta[["filename_clean", "label", "group", "size"]].copy()
    saved["predicted"] = pred
    saved["correct"] = saved["label"].to_numpy() == pred
    saved.to_csv(OUT / f"{name}_predictions.csv", index=False)

print("\nSaved reports to:", OUT)



Independent recordings: 60
Fault-instance groups: 15
Median features: 76
Median + IQR features: 152

========== FEATURE REPRESENTATION COMPARISON ==========


,feature_representation,accuracy,balanced_accuracy,macro_f1,correct,total
1,median_plus_IQR,0.833,0.827,0.820,50.0,60.0
0,median_only,0.783,0.765,0.754,47.0,60.0



========== PERFORMANCE BY DEFECT SIZE ==========


,feature_representation,defect_size,recordings,accuracy
0,median_only,0.007,20,1.000
1,median_only,0.014,12,0.333
2,median_only,0.021,20,0.950
3,median_only,0.028,8,0.500
4,median_plus_IQR,0.007,20,1.000
5,median_plus_IQR,0.014,12,0.667
6,median_plus_IQR,0.021,20,0.900
7,median_plus_IQR,0.028,8,0.500



Saved reports to: /content/drive/MyDrive/BTP_Bearing_Fault_Project/colab_model_diagnostics/median_vs_iqr_v1


## 6. Train and save the final Extra Trees v2 model

In [17]:

# FINAL CANDIDATE V2: EXTRA TREES + MEDIAN AND WINDOW IQR FEATURES
# Reuses existing notebook variables; no feature extraction or model search rerun.

import joblib

required_vars = [
    "X_median_iqr", "y_rec", "candidates",
    "feature_cols", "ROOT", "extractor", "Path", "np", "pd"
]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(
        "Notebook variables are missing: " + ", ".join(missing_vars)
        + ". Rerun the existing setup/feature comparison cell first."
    )

# 1. Fit the selected candidate on all 60 recording-level rows.
# Grouped CV metrics are calculated in the preceding cell; use those actual
# values instead of copying historical metrics into the new artifact.
final_model_v2 = clone(candidates["Extra Trees"])
final_model_v2.fit(X_median_iqr, y_rec)

model_feature_columns = list(X_median_iqr.columns)
base_feature_columns = list(feature_cols)

cv_row = comparison_iqr.loc[
    comparison_iqr["feature_representation"] == "median_plus_IQR"
].iloc[0]

if len(X_median_iqr) != 60 or len(model_feature_columns) != 152:
    raise ValueError(
        f"Unexpected training shape: {X_median_iqr.shape}. "
        "Expected 60 recordings and 152 median-plus-IQR features."
    )

# 2. Save model and feature-construction metadata together.
model_dir = ROOT / "models"
model_dir.mkdir(parents=True, exist_ok=True)

# Do not overwrite the frozen v2 artifact. Save this reproduction separately;
# promote it to the production filename only after comparing its CV results.
model_path_v2 = model_dir / "bearing_fault_extra_trees_median_iqr_v2_reproduced.joblib"

bundle_v2 = {
    "model": final_model_v2,
    "base_feature_columns": base_feature_columns,
    "model_feature_columns": model_feature_columns,
    "classes": sorted(pd.Series(y_rec).unique()),
    "training_recordings": len(X_median_iqr),
    "training_fault_instance_groups": 15,
    "feature_representation": "recording median + per-feature window IQR",
    "validation_protocol": "Leave one complete fault-instance group out",
    "development_accuracy": float(cv_row["accuracy"]),
    "development_balanced_accuracy": float(cv_row["balanced_accuracy"]),
    "development_macro_f1": float(cv_row["macro_f1"]),
    "validation_note": (
        "Development estimate after feature selection. "
        "Independent external validation is still required."
    )
}

joblib.dump(bundle_v2, model_path_v2, compress=3)
bundle_iqr = joblib.load(model_path_v2)

print("Saved model:", model_path_v2)
print("Training recordings:", bundle_iqr["training_recordings"])
print("Input signal features:", len(base_feature_columns))
print("Model features:", len(model_feature_columns))
print("Reload verification:", bundle_iqr["model"].n_features_in_)
print("Grouped CV accuracy (median + IQR):", round(float(cv_row["accuracy"]), 4))
print("Grouped CV balanced accuracy:", round(float(cv_row["balanced_accuracy"]), 4))
print("Grouped CV macro-F1:", round(float(cv_row["macro_f1"]), 4))
print("Original frozen v2 artifact was not overwritten.")

# 3. Predict a new MAT file using matching median + IQR features.
def predict_bearing_mat_iqr(mat_path):
    mat_path = Path(mat_path)

    if not mat_path.is_file() or mat_path.suffix.lower() != ".mat":
        raise FileNotFoundError(f"MAT file not found: {mat_path}")

    signals, rpm = extractor.read_signals_and_rpm(mat_path)
    filters = extractor.make_filter_bank()

    window_rows = []

    for mat_variable, signal in signals:
        if len(signal) < extractor.WINDOW_SAMPLES:
            raise ValueError(f"Signal too short: {mat_path.name}")

        # Match original feature extraction: filter before windowing.
        filtered_bank = {
            name: extractor.sosfiltfilt(sos, signal)
            for name, sos in filters.items()
        }
        envelope_bank = {
            name: np.abs(extractor.hilbert(filtered))
            for name, filtered in filtered_bank.items()
        }

        signal_id = (
            mat_path.stem
            if len(signals) == 1
            else f"{mat_path.stem}__{mat_variable.removesuffix('_DE_time')}"
        )

        n_windows = len(signal) // extractor.WINDOW_SAMPLES

        for window_index in range(n_windows):
            start = window_index * extractor.WINDOW_SAMPLES
            stop = start + extractor.WINDOW_SAMPLES

            row = {}
            for band_name in extractor.BANDS:
                row.update(
                    extractor.extract_band_features(
                        filtered_bank[band_name][start:stop],
                        envelope_bank[band_name][start:stop],
                        rpm,
                        band_name
                    )
                )

            row["signal_id"] = signal_id
            window_rows.append(row)

    if not window_rows:
        raise ValueError("No usable feature windows were extracted.")

    window_df = pd.DataFrame(window_rows)
    base_cols = bundle_iqr["base_feature_columns"]

    missing = [c for c in base_cols if c not in window_df.columns]
    if missing:
        raise ValueError(
            "Feature extractor/model mismatch. Missing: " + ", ".join(missing)
        )

    results = []

    # One prediction per original signal, using median + IQR.
    for signal_id, part in window_df.groupby("signal_id", sort=False):
        features = part[base_cols].replace([np.inf, -np.inf], np.nan)

        medians = features.median()
        iqr = features.quantile(0.75) - features.quantile(0.25)

        row = {c: medians[c] for c in base_cols}
        row.update({f"{c}__window_iqr": iqr[c] for c in base_cols})

        X_new = pd.DataFrame(
            [row], columns=bundle_iqr["model_feature_columns"]
        )

        if X_new.isna().any().any():
            # The saved pipeline handles missing values by imputation.
            pass

        prediction = bundle_iqr["model"].predict(X_new)[0]
        scores = bundle_iqr["model"].predict_proba(X_new)[0]

        result = {
            "filename": mat_path.name,
            "signal_id": signal_id,
            "rpm_used": rpm,
            "windows_evaluated": len(part),
            "predicted_fault": prediction,
            "model_probability_score": float(scores.max())
        }

        for cls, score in zip(bundle_iqr["model"].classes_, scores):
            result[f"score_{cls}"] = float(score)

        results.append(result)

    return pd.DataFrame(results)


print("\nPrediction function ready: predict_bearing_mat_iqr(path)")


Saved model: /content/drive/MyDrive/BTP_Bearing_Fault_Project/models/bearing_fault_extra_trees_median_iqr_v2_reproduced.joblib
Training recordings: 60
Input signal features: 76
Model features: 152
Reload verification: 152
Grouped CV accuracy (median + IQR): 0.8333
Grouped CV balanced accuracy: 0.8274
Grouped CV macro-F1: 0.82
Original frozen v2 artifact was not overwritten.

Prediction function ready: predict_bearing_mat_iqr(path)


## 7. Predict one new recording (optional)

Set `MAT_PATH` to a **new** `.mat` recording, not one used to create the development table. The feature extractor must be able to determine the signal and RPM metadata from the file. If RPM is not encoded in the file and its filename, provide a correct RPM handling rule before trusting the prediction.

In [18]:
# Edit this path to point to a new .mat recording in your Drive project.
MAT_PATH = ROOT / "incoming_mat" / "your_new_recording.mat"

if MAT_PATH.is_file():
    prediction_result = predict_bearing_mat_iqr(MAT_PATH)
    display(prediction_result)
    output_dir = ROOT / "predictions"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{MAT_PATH.stem}_prediction.csv"
    prediction_result.to_csv(output_path, index=False)
    print("Saved prediction:", output_path)
else:
    print("No inference run performed. Place a new .mat file at:", MAT_PATH)


No inference run performed. Place a new .mat file at: /content/drive/MyDrive/BTP_Bearing_Fault_Project/incoming_mat/your_new_recording.mat
